In [1]:
import torch
from medclip import MedCLIPModel, MedCLIPVisionModelViT, MedCLIPProcessor
import chromadb
import pandas as pd
import langchain
import langgraph

print("✅ All imports successful!")
print("Torch version:", torch.__version__)


✅ All imports successful!
Torch version: 2.5.1+cu121


In [2]:
import torch
print(torch.cuda.is_available()) 

True


In [7]:
from medclip import MedCLIPModel, MedCLIPVisionModelViT
from medclip import MedCLIPProcessor
from PIL import Image

# prepare for the demo image and texts
processor = MedCLIPProcessor()
image = Image.open('./PadChest_GR_resized_224_224/251488034557732338959601580328898734705_wigcpj.png')
inputs = processor(
    text=["apical pleural thickening", 
        "costophrenic angle blunting", 
        "Normal",
        "mediastinic lipomatosis"], 
    images=image, 
    return_tensors="pt", 
    padding=True
    )

# pass to MedCLIP model
model = MedCLIPModel(vision_cls=MedCLIPVisionModelViT)
""" model.from_pretrained() """ # downloading pre_trained weights (only the forst time)
model.cuda()
outputs = model(**inputs)
print(outputs.keys())
# dict_keys(['img_embeds', 'text_embeds', 'logits', 'loss_value', 'logits_per_text'])

Some weights of the model checkpoint at microsoft/swin-tiny-patch4-window7-224 were not used when initializing SwinModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing SwinModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing SwinModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at emilyalsentzer/Bio_ClinicalBERT were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.t

dict_keys(['img_embeds', 'text_embeds', 'logits', 'loss_value', 'logits_per_text'])


In [15]:
logits = outputs['logits_per_text']
probs = torch.softmax(logits, dim=0)  # dim=0 car logits_per_text est (n_texts, 1)
for i, txt in enumerate(["apical pleural thickening", "costophrenic angle blunting", "Normal", "mediastinic lipomatosis"]):
    print(f"{txt:30s} -> logit: {logits[i].item():.3f} | prob: {probs[i].item()*100:.1f}%")

apical pleural thickening      -> logit: 0.344 | prob: 25.4%
costophrenic angle blunting    -> logit: 0.570 | prob: 31.8%
Normal                         -> logit: 0.478 | prob: 29.0%
mediastinic lipomatosis        -> logit: -0.259 | prob: 13.9%


In [16]:
processor = MedCLIPProcessor()
image = Image.open('./PadChest_GR_resized_224_224/251488034557732338959601580328898734705_wigcpj.png')
inputs = processor(
    text=["Minimal biapical pleural thickening.", 
        "Slight blunting of the posterior left costophrenic angle.", 
        "Occupation of the right cardiophrenic angle probably related to pericardial fat."], 
    images=image, 
    return_tensors="pt", 
    padding=True
    )

# pass to MedCLIP model
model = MedCLIPModel(vision_cls=MedCLIPVisionModelViT)
model.cuda()
outputs = model(**inputs)
print(outputs.keys())

logits = outputs['logits_per_text']


Some weights of the model checkpoint at microsoft/swin-tiny-patch4-window7-224 were not used when initializing SwinModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing SwinModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing SwinModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at emilyalsentzer/Bio_ClinicalBERT were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.trans

dict_keys(['img_embeds', 'text_embeds', 'logits', 'loss_value', 'logits_per_text'])


In [17]:
probs = torch.softmax(logits, dim=0)  # dim=0 car logits_per_text est (n_texts, 1)
for i, txt in enumerate(["Minimal biapical pleural thickening.", 
        "Slight blunting of the posterior left costophrenic angle.", 
        "Occupation of the right cardiophrenic angle probably related to pericardial fat."]):
    print(f"{txt:30s} -> logit: {logits[i].item():.3f} | prob: {probs[i].item()*100:.1f}%")

Minimal biapical pleural thickening. -> logit: -1.051 | prob: 27.6%
Slight blunting of the posterior left costophrenic angle. -> logit: -0.751 | prob: 37.2%
Occupation of the right cardiophrenic angle probably related to pericardial fat. -> logit: -0.808 | prob: 35.2%


## encoding and storing in the vectorDB


In [42]:
model = MedCLIPModel(vision_cls=MedCLIPVisionModelViT)
model.cuda()

processor = MedCLIPProcessor()

# load image
image = Image.open('./PadChest_GR_resized_224_224/251488034557732338959601580328898734705_wigcpj.png')

# pass a dummy text list (length 1, because batch size must match)
inputs = processor(
    images=image,
    text=[""],  # dummy text
    return_tensors="pt",
    padding=True
)
inputs = {k: v.cuda() for k, v in inputs.items()}

# forward pass
with torch.no_grad():
    outputs = model(**inputs)

img_embeds = outputs['img_embeds']  # shape [1, embedding_dim]
img_embeds = img_embeds.cpu().numpy()

img_embeds

c:\Users\ahmed\OneDrive\Desktop\tutore\.tutore_venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at microsoft/swin-tiny-patch4-window7-224 were not used when initializing SwinModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing SwinModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing SwinModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
c:\Users\ahmed\OneDrive\Desktop\tutore\.tutore_venv\Lib\site-packages\t

array([[-7.46267987e-03, -2.81946100e-02, -5.72954603e-02,
        -1.00782828e-03,  1.36164622e-02, -2.48876493e-02,
        -3.46142277e-02,  3.44081484e-02, -5.39135113e-02,
        -2.90571731e-02, -4.61796820e-02, -9.20775384e-02,
         3.28523777e-02, -5.23243695e-02, -2.54719984e-03,
         6.26708055e-03,  2.38162819e-02, -6.76582083e-02,
         7.91198611e-02, -5.79326879e-03,  2.38514859e-02,
         1.35391764e-02,  3.57264988e-02,  4.09528688e-02,
         5.97974621e-02,  1.20658107e-01,  4.76308316e-02,
        -1.59559082e-02,  1.03816859e-01,  4.65591997e-02,
        -2.13734265e-02, -1.06848087e-02,  4.75542843e-02,
         4.68695946e-02,  8.16742256e-02,  8.05913955e-02,
         1.83818229e-02,  6.07452691e-02,  1.47180976e-02,
        -1.00182600e-01, -3.37776132e-02,  2.27649584e-02,
         5.71364984e-02,  5.28760292e-02, -1.70817249e-03,
         3.71164009e-02, -4.23813192e-03,  1.49126537e-02,
         1.31210906e-03, -2.60704793e-02, -2.06606872e-0

### Encode test images with MedCLIP and store in Chroma (server mode)
The following cells will:
- Load the test split and aggregate `label` and `sentence_en` per `ImageID`.
- Encode each test image using MedCLIP (same pattern as the previous single-image snippet).
- Upsert embeddings + metadata into a Chroma collection running at `localhost:8000` (HTTP server mode).


In [3]:
import os
import math
import pandas as pd
import numpy as np
import torch
from PIL import Image
import chromadb
from chromadb import PersistentClient

from medclip import MedCLIPModel, MedCLIPVisionModelViT, MedCLIPProcessor
from peft import PeftModel

#=====================
# CONFIG
#=====================
TRAIN_CSV = './tutore_train/train/train_metadata.csv'
TEST_CSV  = './tutore_test/test/test_metadata.csv'

PERSIST_DIR = 'vectorDB'
COLLECTION_NAME = 'padchest_sentences_medclip'

BATCH_SIZE = 64   # texts per batch


#=====================
# LOAD DATA
#=====================
assert os.path.exists(TRAIN_CSV), f"{TRAIN_CSV} not found"
assert os.path.exists(TEST_CSV), f"{TEST_CSV} not found"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# Keep only valid English sentences
df = df[df['findings'].notna()]

print(f"Total sentences loaded: {len(df)}")


#=====================
# INIT MEDCLIP PROCESSOR & MODEL
#=====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = MedCLIPProcessor()

# Load base model
model = MedCLIPModel(vision_cls=MedCLIPVisionModelViT)
model.from_pretrained()

# Load LoRA fine-tuned weights
lora_path = "./lora_findings/lora"
model = PeftModel.from_pretrained(model, lora_path)

model.to(device)
model.eval()

print("MedCLIP model (fine-tuned) loaded.")


#=====================
# CHROMA CLIENT
#=====================
os.makedirs(PERSIST_DIR, exist_ok=True)
client = PersistentClient(path=PERSIST_DIR)

collection = client.get_or_create_collection(
    name="cxr_findings",
    metadata={
        "hnsw:space": "cosine"
    }
)

print(f"Using Chroma collection: {collection.name}")

Total sentences loaded: 30633


c:\Users\ahmed\OneDrive\Desktop\tutore\.tutore_venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\ahmed\OneDrive\Desktop\tutore\.tutore_venv\Lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Some weights of the model checkpoint at microsoft/swin-tiny-patch4-window7-224 were not used when initializing SwinModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing SwinModel from the checkpoint of a model trained on another task or with anoth

load model weight from: ./pretrained/medclip-vit


c:\Users\ahmed\OneDrive\Desktop\tutore\.tutore_venv\Lib\site-packages\peft\utils\save_and_load.py:198: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  adapters_weights = torch

MedCLIP model (fine-tuned) loaded.
Using Chroma collection: cxr_findings


In [6]:
#=====================
# TEXT ENCODING (BATCHED + NORMALIZED)
#=====================

def encode_single_text_with_medclip(text):
    """
    Encode a single finding using MedCLIP.
    Returns a 1D normalized embedding.
    """
    model.eval()
    with torch.no_grad():
        inputs = processor(
            text=[text],
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        text_embeds = model.encode_text(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        emb = text_embeds[0]

        if emb.dim() > 1:
            emb = emb[0]

        emb = emb.cpu().numpy()

        norm = np.linalg.norm(emb)
        if norm > 0:
            emb = emb / norm

        return emb



#=====================
# PREPARE TRAIN DATA ONLY (NO LEAKAGE)
#=====================

train_df = train_df[train_df["findings"].notna()].reset_index(drop=True)

texts     = train_df["findings"].tolist()
image_ids = train_df["image_id"].tolist()

ids = [f"{image_ids[i]}_{i}" for i in range(len(texts))]

num_items   = len(texts)
num_batches = math.ceil(num_items / BATCH_SIZE)

print(f"Encoding and storing {num_items} TRAIN findings in {num_batches} batches...")


#=====================
# BATCH ENCODE + UPSERT INTO CHROMA
#=====================

for b in range(num_batches):
    start = b * BATCH_SIZE
    end   = min((b + 1) * BATCH_SIZE, num_items)

    ids_batch        = []
    embeddings_batch = []
    metadatas_batch  = []
    documents_batch  = []

    for i in range(start, end):
        txt = texts[i]

        emb = encode_single_text_with_medclip(txt)

        ids_batch.append(ids[i])
        embeddings_batch.append(emb.tolist())
        documents_batch.append(txt)

        metadatas_batch.append({
            "image_id": image_ids[i],
            "split": "train"
        })

    collection.upsert(
        ids=ids_batch,
        embeddings=embeddings_batch,
        metadatas=metadatas_batch,
        documents=documents_batch
    )

    print(f"Batch {b+1}/{num_batches} stored ({len(ids_batch)} items)")

print("\n✅ DONE — TRAIN findings embeddings safely stored in Chroma.")


Encoding and storing 24506 TRAIN findings in 383 batches...
Batch 1/383 stored (64 items)
Batch 2/383 stored (64 items)
Batch 3/383 stored (64 items)
Batch 4/383 stored (64 items)
Batch 5/383 stored (64 items)
Batch 6/383 stored (64 items)
Batch 7/383 stored (64 items)
Batch 8/383 stored (64 items)
Batch 9/383 stored (64 items)
Batch 10/383 stored (64 items)
Batch 11/383 stored (64 items)
Batch 12/383 stored (64 items)
Batch 13/383 stored (64 items)
Batch 14/383 stored (64 items)
Batch 15/383 stored (64 items)
Batch 16/383 stored (64 items)
Batch 17/383 stored (64 items)
Batch 18/383 stored (64 items)
Batch 19/383 stored (64 items)
Batch 20/383 stored (64 items)
Batch 21/383 stored (64 items)
Batch 22/383 stored (64 items)
Batch 23/383 stored (64 items)
Batch 24/383 stored (64 items)
Batch 25/383 stored (64 items)
Batch 26/383 stored (64 items)
Batch 27/383 stored (64 items)
Batch 28/383 stored (64 items)
Batch 29/383 stored (64 items)
Batch 30/383 stored (64 items)
Batch 31/383 stored

In [8]:
# Quick verification: count stored items and fetch a sample
count = collection.count()
print(f"Collection '{COLLECTION_NAME}' now contains {count} embeddings.")

res = collection.get(limit=5)
for i in range(len(res["ids"])):
    print(res["ids"][i], "→", res["documents"][i][:80])


Collection 'padchest_sentences_medclip' now contains 24506 embeddings.
img_30118.png_0 → A right-sided chest tube remains in unchanged position. There is a small right-s
img_22815.png_1 → Cardiac, mediastinal and hilar contours are normal. Lungs are clear and the pulm
img_14620.png_2 →  Transesophageal tube courses below the diaphragm and out of view. Left subclavi
img_24425.png_3 → The cardiac, mediastinal, and hilar contours are normal. There is no mediastinal
img_6418.png_4 → There is no consolidation, pleural effusion, or pneumothorax. There is no pulmon


## Testing with new image



In [9]:
def encode_image_with_medclip(image_path):
    """
    Encode a single CXR image using MedCLIP.
    Returns a 1D L2-normalized embedding.
    """
    model.eval()
    with torch.no_grad():
        image = Image.open(image_path).convert("RGB")

        inputs = processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = inputs["pixel_values"].to(device)

        image_embeds = model.encode_image(pixel_values)

        emb = image_embeds[0]

        if emb.dim() > 1:
            emb = emb[0]

        emb = emb.cpu().numpy()

        norm = np.linalg.norm(emb)
        if norm > 0:
            emb = emb / norm

        return emb


In [10]:
def retrieve_findings_for_image(
    image_path,
    collection,
    top_k=10
):
    """
    Retrieve top-K similar FINDINGS for a given CXR image.
    """
    image_emb = encode_image_with_medclip(image_path)

    results = collection.query(
        query_embeddings=[image_emb.tolist()],
        n_results=top_k,
        where={"split": "train"}  # safety guard
    )

    retrieved = []

    for i in range(len(results["documents"][0])):
        retrieved.append({
            "finding": results["documents"][0][i],
            "distance": results["distances"][0][i],
            "image_id": results["metadatas"][0][i]["image_id"]
        })

    return retrieved


In [15]:
def filter_by_distance(retrieved_findings, max_distance=0.6):
    print("\n[STEP 1] Distance filtering")
    print(f"Threshold: {max_distance}")

    filtered = []
    for f in retrieved_findings:
        print(f"  - {f['distance']:.3f} | {f['finding'][:80]}...")
        if f["distance"] <= max_distance:
            filtered.append(f)

    print(f"Kept {len(filtered)} / {len(retrieved_findings)} findings")
    return filtered

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity # removing duplicates (semantically identical)

## Re-embed each finding (using MedCLIP text encoder)

## Compare embeddings using cosine similarity

## Decide duplication based on meaning, not words

def deduplicate_findings(findings, similarity_threshold=0.9):
    print("\n[STEP 2] Semantic deduplication")
    print(f"Similarity threshold: {similarity_threshold}")

    texts = [f["finding"] for f in findings]
    embeddings = [encode_single_text_with_medclip(t) for t in texts]

    kept = []
    kept_embeddings = []

    for i, (text, emb) in enumerate(zip(texts, embeddings)):
        is_duplicate = False

        for kept_emb in kept_embeddings:
            sim = cosine_similarity(
                emb.reshape(1, -1),
                kept_emb.reshape(1, -1)
            )[0][0]

            if sim >= similarity_threshold:
                print(f"  DUPLICATE (sim={sim:.3f}): {text[:80]}...")
                is_duplicate = True
                break

        if not is_duplicate:
            print(f"  KEPT: {text[:80]}...")
            kept.append(text)
            kept_embeddings.append(emb)

    print(f"Deduplicated: {len(texts)} → {len(kept)} findings")
    return kept


In [ ]:
from sklearn.cluster import AgglomerativeClustering

# --------------------------------------------------
# STEP 3: Semantic clustering of findings
# --------------------------------------------------
# Goal:
# Group clinically related (but non-duplicate) findings together
# e.g. pleural effusion findings, device-related findings, pneumothorax findings.
#
# Method:
# - Re-embed each finding using MedCLIP
# - Cluster using cosine distance (1 - cosine similarity)
# - Agglomerative clustering is used to avoid fixing the number of clusters
#
# Interpretation:
# - distance_threshold ~ 0.25  → cosine similarity ≥ 0.75
# - Findings within a cluster describe the same type of abnormality
#
# This step organizes findings before consolidation and generation,
# mimicking how a radiologist mentally groups observations.
# --------------------------------------------------

def cluster_findings(findings, distance_threshold=0.25):
    print("\n[STEP 3] Clustering findings")
    print(f"Distance threshold: {distance_threshold}")

    embeddings = [encode_single_text_with_medclip(f) for f in findings]

    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric="cosine",
        linkage="average"
    )

    labels = clustering.fit_predict(embeddings)

    clusters = {}
    for label, finding in zip(labels, findings):
        clusters.setdefault(label, []).append(finding)

    for label, items in clusters.items():
        print(f"\n  Cluster {label} ({len(items)} items):")
        for it in items:
            print(f"   - {it[:80]}...")

    return clusters


In [18]:
def produce_compact_findings(clusters):
    print("\n[STEP 4] Producing compact findings")

    compact = []

    for label, items in clusters.items():
        if len(items) == 1:
            compact.append(items[0])
            print(f"  Cluster {label}: single → kept")
            continue

        embeddings = [encode_single_text_with_medclip(t) for t in items]
        sims = cosine_similarity(embeddings)

        avg_sims = sims.mean(axis=1)
        best_idx = avg_sims.argmax()

        chosen = items[best_idx]
        print(f"  Cluster {label}: selected representative")
        print(f"    → {chosen[:80]}...")

        compact.append(chosen)

    print(f"\nFinal compact findings count: {len(compact)}")
    return compact


In [21]:
def format_compact_findings(findings):
    result = ""
    for i, text in enumerate(findings, start=1):
        result += f"{i}. {text.strip()}\n"
    return result.strip()

## feeding context to LLM

In [26]:
image_path = "./tutore_test/test/images_test/img_25248.png"

retrieved_findings = retrieve_findings_for_image(
    image_path=image_path,
    collection=collection,
    top_k=10
)

for r in retrieved_findings:
    print(f"[{r['distance']:.3f}] {r['finding']}")

[0.354] Since prior study, there has been no interval change in position of right chest wall Port-A-Cath, terminating in the upper right atrium, as well as a left chest wall pulse generator, with dual lead pacing wires terminating in the right atrium and right ventricle. Median sternotomy wires are intact. A right pleural effusion has slightly increased compared to the prior study, along with fluid tracking along the horizontal fissure on the right, and subsegmental atelectasis in the right lung base. Left basilar atelectasis is also increased, as has a small left pleural effusion. There is no pneumothorax. Biapical pleural thickening is stable. The overall heart size is unchanged. 
[0.365] There has been interval placement of a left pigtail pleural catheter. A small left apical pneumothorax is noted. The left pacemaker positioning is unchanged. 
[0.387] A left-sided pacemaker is again seen with leads terminating in the right atrium and right ventricle. Median sternotomy wires appear i

In [27]:
filtered = filter_by_distance(retrieved_findings, max_distance=0.6)
deduped  = deduplicate_findings(filtered, similarity_threshold=0.9)
clusters = cluster_findings(deduped, distance_threshold=0.25)
compact_findings = produce_compact_findings(clusters)
ranked_findings = format_compact_findings(compact_findings)


[STEP 1] Distance filtering
Threshold: 0.6
  - 0.354 | Since prior study, there has been no interval change in position of right chest ...
  - 0.365 | There has been interval placement of a left pigtail pleural catheter. A small le...
  - 0.387 | A left-sided pacemaker is again seen with leads terminating in the right atrium ...
  - 0.387 | A right PICC line, right chest tube and left chest wall dual lead pacemaker are ...
  - 0.388 | Since the most recent examination, the patient has been extubated. A right-sided...
  - 0.391 | The right-sided chest tube and central chest tube have been removed. Right-sided...
  - 0.392 | Left-sided PICC now appears to be terminating in the left atrium, which may be d...
  - 0.394 | The patient has had prior aortic valve replacement with intact and aligned stern...
  - 0.398 | Compared with the prior film, multiple lines and tubes have been removed, includ...
  - 0.401 | Since the prior exam, the right-sided chest tube has been removed. There is no p

In [29]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",  
    temperature=0.7
)


In [30]:
system_message = """
You are a radiology assistant.

You are given a list of factual FINDINGS extracted from similar chest X-ray cases.
These findings are already relevant and non-contradictory.

TASK:
- Rewrite the findings into a single, concise FINDINGS section.
- Merge overlapping statements.
- Preserve all factual information.
- Do NOT add new abnormalities.
- Do NOT infer diagnoses.
- Do NOT mention prior studies unless explicitly stated.

STYLE:
- Objective radiology language
- Past tense
- No bullet points
- No speculation

FINDINGS:
{findings}
"""


report = llm.invoke([
    SystemMessage(content=system_message.format(
            findings= ranked_findings , 
        ))
])

report.content

'The right PICC line, left chest‑wall dual‑lead pacemaker, right internal jugular central venous catheter, and left pigtail catheter were all present. A right hydro pneumothorax had been noted on the prior exam and was mildly decreased in size; it was absent after removal of the right chest tube. A small left pleural effusion was present on the prior exam and was absent on the current study. Minimal left basilar atelectasis was noted, unchanged from the prior exam. The lungs were otherwise clear. The cardiomediastinal silhouette was normal and unchanged.'

## Computing cosine similarity

In [31]:
from sklearn.metrics.pairwise import cosine_similarity

def image_text_alignment_score(image_path, generated_text):
    """
    Compute cosine similarity between image and generated findings.
    Returns (cosine_score, percentage_score).
    """
    img_emb  = encode_image_with_medclip(image_path)
    text_emb = encode_single_text_with_medclip(generated_text)

    cosine_score = cosine_similarity(
        img_emb.reshape(1, -1),
        text_emb.reshape(1, -1)
    )[0][0]

    percentage_score = cosine_score * 100

    return cosine_score, percentage_score


In [32]:
cos_sim, percent = image_text_alignment_score(
    image_path=image_path,
    generated_text=report.content
)

print(f"Image–Text Cosine Similarity: {cos_sim:.3f}")
print(f"Alignment Score: {percent:.1f}%")

Image–Text Cosine Similarity: 0.589
Alignment Score: 58.9%
